In [ ]:
import os, math, shutil
import rasterio
from rasterio.windows import Window
from shapely.geometry import box
import geopandas as gpd
import numpy as np
from random import choice

def preprocess_dataset(patch_size, paths, shapefile_path, train_dir, test_dir, test_split=0.2):
    # Load raw raster files
    naip_data = rasterio.open(paths["naip"])
    mask_data = rasterio.open(paths["mask"])
    hillshade_data = rasterio.open(paths["hillshade"])
    dem_data = rasterio.open(paths["dem"])
    
    naip = naip_data.read()
    naip_rows = math.floor(naip.shape[1] / patch_size)
    naip_cols = math.floor(naip.shape[2] / patch_size)

    def create_patches(data_obj, out_subdir, tag):
        data = data_obj.read()
        rows = math.floor(data.shape[1] / patch_size)
        cols = math.floor(data.shape[2] / patch_size)

        for row in range(rows):
            y_offset = row * patch_size
            for col in range(cols):
                x_offset = col * patch_size
                window = Window(x_offset, y_offset, patch_size, patch_size)
                patch = data_obj.read(window=window)

                patch_meta = data_obj.meta.copy()
                patch_meta.update({
                    "height": patch_size,
                    "width": patch_size,
                    "transform": rasterio.windows.transform(window, data_obj.transform)
                })

                patch[patch == 1.79e+308] = 0  # Set nodata to 0
                patch_name = f"SCC_{tag}_patch{row}-{col}.tif"
                patch_path = os.path.join(out_subdir, patch_name)

                os.makedirs(out_subdir, exist_ok=True)
                with rasterio.open(patch_path, "w", **patch_meta) as dst:
                    dst.write(patch)

    print("Creating patches...")
    create_patches(naip_data, os.path.join(train_dir, "images"), "NAIP_1m")
    create_patches(mask_data, os.path.join(train_dir, "masks"), "mask")
    create_patches(hillshade_data, os.path.join(train_dir, "hillshade"), "hillshade")
    create_patches(dem_data, os.path.join(train_dir, "dem"), "dem")

    print("Removing out-of-bound patches...")
    shp = gpd.read_file(shapefile_path)
    for row in range(naip_rows):
        for col in range(naip_cols):
            patch_id = f"{row}-{col}"
            files = {
                "img": f"{train_dir}/images/SCC_NAIP_1m_patch{patch_id}.tif",
                "mask": f"{train_dir}/masks/SCC_mask_patch{patch_id}.tif",
                "hillshade": f"{train_dir}/hillshade/SCC_hillshade_patch{patch_id}.tif",
                "dem": f"{train_dir}/dem/SCC_dem_patch{patch_id}.tif"
            }

            if not all(os.path.exists(f) for f in files.values()):
                continue

            with rasterio.open(files["img"]) as src:
                raster_crs = src.crs
                bounds = src.bounds
                raster_bbox = box(bounds.left, bounds.bottom, bounds.right, bounds.top)

                shp_proj = shp.to_crs(raster_crs) if shp.crs != raster_crs else shp
                if not raster_bbox.intersects(shp_proj.unary_union):
                    for f in files.values():
                        os.remove(f)

    print("Splitting into train/test...")
    image_files = os.listdir(os.path.join(train_dir, "images"))
    total = len(image_files)
    split_count = int(test_split * total)
    picked = set()

    while len(picked) < split_count:
        filename = choice(image_files)
        patch_id = filename.split("patch")[-1].replace(".tif", "")
        if patch_id not in picked:
            picked.add(patch_id)

    for patch_id in picked:
        file_map = {
            "images": f"SCC_NAIP_1m_patch{patch_id}.tif",
            "masks": f"SCC_mask_patch{patch_id}.tif",
            "hillshade": f"SCC_hillshade_patch{patch_id}.tif",
            "dem": f"SCC_dem_patch{patch_id}.tif"
        }
        for subdir, filename in file_map.items():
            src = os.path.join(train_dir, subdir, filename)
            dst_dir = os.path.join(test_dir, subdir)
            os.makedirs(dst_dir, exist_ok=True)
            dst = os.path.join(dst_dir, filename)
            if os.path.exists(src):
                shutil.copy2(src, dst)
                os.remove(src)

    print("Done!")

